In [2]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from apt_detection.device_telemetry.features import main_features

PROC = Path("../../data/processed/device_telemetry")
F2 = pd.read_parquet(PROC / "phase2_features.parquet")
FEATS = main_features()

F2["kind"] = np.where(F2["label"] == 0, "normal",
                      np.where(F2["flood"] == 1, "flood", "other"))

CUTOFF = pd.Timestamp("2023-12-03 12:00", tz="UTC").timestamp()
dev = F2[F2["t_start"] < CUTOFF].copy()
test = F2[F2["t_start"] >= CUTOFF].copy()

for name, part in [("development", dev), ("held-out test", test)]:
    print(name, len(part), part["kind"].value_counts().to_dict())
    print("   tactics:", part.loc[part["label"] == 1, "tactic"].value_counts().to_dict())

test.to_parquet(PROC / "phase2_heldout_test.parquet")

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
development 10884 {'flood': 7049, 'normal': 3766, 'other': 69}
   tactics: {'collection': 7063, 'discovery': 21, 'exfiltration': 14, 'CandC': 11, 'persistence': 5, 'credentialAccess': 3, 'initialAccess': 1}
held-out test 1559 {'normal': 1516, 'other': 36, 'flood': 7}
   tactics: {'defenceEvasion': 22, 'credentialAccess': 8, 'lateralMovement': 6, 'collection': 4, 'discovery': 3}


In [3]:
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
folds = list(cv.split(dev[FEATS], dev["kind"], dev["time_block"]))

for i, (tr, te) in enumerate(folds):
    k = dev["kind"].iloc[te].value_counts()
    print(f"fold {i}: test={len(te):5d}  normal={k.get('normal', 0):5d}  "
          f"flood={k.get('flood', 0):5d}  other={k.get('other', 0):3d}  "
          f"hour blocks={dev['time_block'].iloc[te].nunique()}")

fold 0: test= 4665  normal=  194  flood= 4457  other= 14  hour blocks=7
fold 1: test= 2836  normal=  254  flood= 2566  other= 16  hour blocks=9
fold 2: test= 1349  normal= 1328  flood=    8  other= 13  hour blocks=8
fold 3: test= 1152  normal= 1129  flood=   10  other= 13  hour blocks=7
fold 4: test=  882  normal=  861  flood=    8  other= 13  hour blocks=7


In [4]:
from apt_detection.device_telemetry.evaluate import score_report

y_dev, kind_dev = dev["label"].to_numpy(), dev["kind"].to_numpy()

rule = (dev["prog_file_op"].eq(1) | dev["cmd_hidden_path"].eq(1)).astype(int)
baselines = {
    "always normal": np.zeros(len(dev)),
    "always attack": np.ones(len(dev)),
    "one-line rule": rule.to_numpy(),
}
results = pd.DataFrame({name: score_report(y_dev, s, kind_dev) for name, s in baselines.items()}).T
results.round(3)

,recall_all,recall_flood,recall_other,fpr,precision,pr_auc_other
always normal,0.000,0.000,0.000,0.000,0.000,0.018
always attack,1.000,1.000,1.000,1.000,0.654,0.018
one-line rule,0.985,0.995,0.043,0.045,0.976,0.018


In [5]:
Path("../../docs/results").mkdir(parents=True, exist_ok=True)
results.round(3).to_csv("../../docs/results/baselines_dev.csv")

In [6]:
from lightgbm import LGBMClassifier
from apt_detection.device_telemetry.train import cv_scores

X_dev = dev[FEATS]

def make_lgbm():
    return LGBMClassifier(n_estimators=200, learning_rate=0.05, num_leaves=15,
                          min_child_samples=10, random_state=42, verbose=-1)

oof = {}
oof["LightGBM (no weights)"] = cv_scores(make_lgbm, X_dev, y_dev, kind_dev, folds, weighted=False)
oof["LightGBM (weighted)"]   = cv_scores(make_lgbm, X_dev, y_dev, kind_dev, folds, weighted=True)

model_results = pd.DataFrame({n: score_report(y_dev, s, kind_dev) for n, s in oof.items()}).T
pd.concat([results, model_results]).round(3)

,recall_all,recall_flood,recall_other,fpr,precision,pr_auc_other
always normal,0.000,0.000,0.000,0.000,0.000,0.018
always attack,1.000,1.000,1.000,1.000,0.654,0.018
one-line rule,0.985,0.995,0.043,0.045,0.976,0.018
LightGBM (no weights),0.999,1.000,0.942,0.010,0.995,0.839
LightGBM (weighted),0.998,0.999,0.899,0.103,0.948,0.316


In [7]:
model_results.round(3).to_csv("../../docs/results/lightgbm_dev.csv")

In [8]:
from apt_detection.device_telemetry.features import FEATURE_GROUPS

def run(feature_list):
    s = cv_scores(make_lgbm, dev[feature_list], y_dev, kind_dev, folds, weighted=False)
    return score_report(y_dev, s, kind_dev)

main_groups = [g for g in FEATURE_GROUPS if g != "artefact"]
ablation = {"all features": run(FEATS)}
for g in main_groups:
    kept = [f for f in FEATS if f not in FEATURE_GROUPS[g]]
    ablation[f"without {g}"] = run(kept)

suspect = FEATURE_GROUPS["cmdline"] + FEATURE_GROUPS["privilege"] + FEATURE_GROUPS["context"]
ablation["behaviour only (no cmdline, privilege, context)"] = run([f for f in FEATS if f not in suspect])

ablation_df = pd.DataFrame(ablation).T[["recall_other", "fpr", "pr_auc_other", "recall_flood"]]
ablation_df.round(3)

,recall_other,fpr,pr_auc_other,recall_flood
all features,0.942,0.010,0.839,1.000
without operations,0.899,0.010,0.835,1.000
without touch,0.942,0.010,0.839,1.000
without privilege,0.899,0.005,0.836,1.000
without program,0.928,0.010,0.847,1.000
without cmdline,0.928,0.031,0.385,0.638
without rarity,0.942,0.002,0.882,1.000
without family,0.957,0.147,0.797,1.000
without context,0.928,0.010,0.814,1.000
"behaviour only (no cmdline, privilege, context)",0.478,0.040,0.263,0.996


In [9]:
ablation_df.round(3).to_csv("../../docs/results/lightgbm_ablation_dev.csv")

In [10]:
single = {"all features": ablation["all features"]}
for f in FEATURE_GROUPS["cmdline"] + FEATURE_GROUPS["privilege"]:
    single[f"without {f}"] = run([x for x in FEATS if x != f])

single_df = pd.DataFrame(single).T[["recall_other", "fpr", "pr_auc_other", "recall_flood"]]
single_df.round(3)

,recall_other,fpr,pr_auc_other,recall_flood
all features,0.942,0.010,0.839,1.000
without cmd_len,0.928,0.010,0.813,1.000
without cmd_pipes,0.942,0.010,0.838,1.000
without cmd_redirect,0.942,0.010,0.833,1.000
without cmd_encoded,0.913,0.010,0.809,1.000
without cmd_hidden_path,0.899,0.025,0.365,0.638
without is_root,0.899,0.005,0.836,1.000
without uid_ne_euid,0.942,0.010,0.839,1.000


In [13]:
single_df.round(3).to_csv("../../docs/results/lightgbm_single_feature_ablation_dev.csv")

In [15]:
import time, io, joblib
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier,
                              AdaBoostClassifier, IsolationForest)
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import OneClassSVM
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from apt_detection.device_telemetry.train import cv_anomaly_scores

suspect = FEATURE_GROUPS["cmdline"] + FEATURE_GROUPS["privilege"] + FEATURE_GROUPS["context"]
FEATURE_SETS = {"full": FEATS, "behaviour": [f for f in FEATS if f not in suspect]}

SUPERVISED = {
    "Random Forest": lambda: RandomForestClassifier(n_estimators=300, min_samples_leaf=3,
                                                    n_jobs=-1, random_state=42),
    "Extra Trees":   lambda: ExtraTreesClassifier(n_estimators=300, min_samples_leaf=3,
                                                  n_jobs=-1, random_state=42),
    "LightGBM":      make_lgbm,
    "AdaBoost":      lambda: AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=2),
                                                n_estimators=200, learning_rate=0.5,
                                                random_state=42),
}
UNSUPERVISED = {
    "Isolation Forest":       lambda: IsolationForest(n_estimators=200, random_state=42, n_jobs=-1),
    "Isolation Forest (log)": lambda: make_pipeline(FunctionTransformer(np.log1p),
                                  IsolationForest(n_estimators=200, random_state=42, n_jobs=-1)),
    "One-Class SVM (log+scaled)": lambda: make_pipeline(FunctionTransformer(np.log1p),
                                  StandardScaler(), OneClassSVM(nu=0.05, gamma="scale")),
}

def model_size_kb(model):
    buf = io.BytesIO()
    joblib.dump(model, buf)
    return buf.tell() / 1024

rows, oof = {}, {}
for set_name, cols in FEATURE_SETS.items():
    Xs = dev[cols]
    for name, make in {**SUPERVISED, **UNSUPERVISED}.items():
        start = time.perf_counter()
        if name in SUPERVISED:
            s = cv_scores(make, Xs, y_dev, kind_dev, folds, weighted=False)
            full_model = make().fit(Xs, y_dev)
        else:
            s = cv_anomaly_scores(make, Xs, kind_dev, folds)
            full_model = make().fit(Xs[kind_dev == "normal"])
        seconds = time.perf_counter() - start
        key = f"{name} | {set_name}"
        oof[key] = s
        rows[key] = {**score_report(y_dev, s, kind_dev),
                     "cv_time_s": round(seconds, 1),
                     "size_kb": round(model_size_kb(full_model), 1)}
        print("done:", key)

screening = pd.DataFrame(rows).T
screening = screening[["recall_other", "fpr", "pr_auc_other", "recall_flood",
                       "precision", "cv_time_s", "size_kb"]]
screening.round(3)

done: Random Forest | full
done: Extra Trees | full
done: LightGBM | full
done: AdaBoost | full
done: Isolation Forest | full
done: Isolation Forest (log) | full
done: One-Class SVM (log+scaled) | full
done: Random Forest | behaviour
done: Extra Trees | behaviour
done: LightGBM | behaviour
done: AdaBoost | behaviour
done: Isolation Forest | behaviour
done: Isolation Forest (log) | behaviour
done: One-Class SVM (log+scaled) | behaviour


,recall_other,fpr,pr_auc_other,recall_flood,precision,cv_time_s,size_kb
Random Forest | full,0.870,0.007,0.913,1.000,0.996,8.9,1316.8
Extra Trees | full,0.899,0.001,0.950,1.000,1.000,5.2,1642.0
LightGBM | full,0.942,0.010,0.839,1.000,0.995,1.0,356.2
AdaBoost | full,0.942,0.001,0.890,1.000,1.000,16.8,187.4
Isolation Forest | full,0.029,0.010,0.056,0.000,0.049,3.4,1285.4
Isolation Forest (log) | full,0.043,0.012,0.093,0.000,0.064,3.3,1462.1
One-Class SVM (log+scaled) | full,0.913,0.106,0.265,1.000,0.947,0.8,85.8
Random Forest | behaviour,0.159,0.030,0.141,0.996,0.984,6.4,1050.4
Extra Trees | behaviour,0.188,0.028,0.129,0.996,0.985,4.8,1136.3
LightGBM | behaviour,0.478,0.040,0.263,0.996,0.979,0.8,356.3


In [16]:
screening.round(3).to_csv("../../docs/results/model_screening_dev.csv")

In [17]:
from apt_detection.device_telemetry.train import cv_stacked_scores

SELECTED = {
    "LightGBM":      (make_lgbm, "supervised"),
    "Extra Trees":   (SUPERVISED["Extra Trees"], "supervised"),
    "One-Class SVM": (UNSUPERVISED["One-Class SVM (log+scaled)"], "anomaly"),
}

ens_rows = {}
for set_name, cols in FEATURE_SETS.items():
    lgbm = oof[f"LightGBM | {set_name}"]
    et   = oof[f"Extra Trees | {set_name}"]
    svm  = oof[f"One-Class SVM (log+scaled) | {set_name}"]

    soft     = (lgbm + et) / 2
    majority = (((lgbm >= 0.5).astype(int) + (et >= 0.5).astype(int) + (svm >= 0.5).astype(int)) >= 2).astype(float)
    stacked  = cv_stacked_scores(SELECTED, dev[cols], y_dev, kind_dev, dev["time_block"], folds)
    oof[f"Stacked | {set_name}"] = stacked

    for name, s in [("Extra Trees (best single)", et), ("LightGBM", lgbm),
                    ("Soft vote (LGBM+ET)", soft), ("Majority vote (2 of 3)", majority),
                    ("Stacked (nested)", stacked)]:
        ens_rows[f"{name} | {set_name}"] = score_report(y_dev, s, kind_dev)
    print("done:", set_name)

ensemble = pd.DataFrame(ens_rows).T[["recall_other", "fpr", "pr_auc_other", "recall_flood", "precision"]]
ensemble.round(3)

done: full
done: behaviour


,recall_other,fpr,pr_auc_other,recall_flood,precision
Extra Trees (best single) | full,0.899,0.001,0.950,1.000,1.000
LightGBM | full,0.942,0.010,0.839,1.000,0.995
Soft vote (LGBM+ET) | full,0.942,0.010,0.952,1.000,0.995
Majority vote (2 of 3) | full,0.913,0.006,0.670,1.000,0.997
Stacked (nested) | full,0.913,0.005,0.900,1.000,0.998
Extra Trees (best single) | behaviour,0.188,0.028,0.129,0.996,0.985
LightGBM | behaviour,0.478,0.040,0.263,0.996,0.979
Soft vote (LGBM+ET) | behaviour,0.348,0.030,0.172,0.996,0.984
Majority vote (2 of 3) | behaviour,0.348,0.027,0.077,0.996,0.986
Stacked (nested) | behaviour,0.232,0.043,0.110,0.001,0.138


In [18]:
non_flood = kind_dev != "flood"
for set_name in FEATURE_SETS:
    scores = pd.DataFrame({
        "LightGBM": oof[f"LightGBM | {set_name}"][non_flood],
        "Extra Trees": oof[f"Extra Trees | {set_name}"][non_flood],
        "One-Class SVM": oof[f"One-Class SVM (log+scaled) | {set_name}"][non_flood],
    })
    print(f"\n{set_name}: rank correlation between model scores (normal + other attacks)")
    print(scores.corr(method="spearman").round(2))


full: rank correlation between model scores (normal + other attacks)
               LightGBM  Extra Trees  One-Class SVM
LightGBM           1.00         0.54           0.66
Extra Trees        0.54         1.00           0.34
One-Class SVM      0.66         0.34           1.00

behaviour: rank correlation between model scores (normal + other attacks)
               LightGBM  Extra Trees  One-Class SVM
LightGBM           1.00         0.51           0.44
Extra Trees        0.51         1.00          -0.16
One-Class SVM      0.44        -0.16           1.00


In [19]:
ensemble.round(3).to_csv("../../docs/results/ensemble_dev.csv")

In [20]:
soft_full = (oof["LightGBM | full"] + oof["Extra Trees | full"]) / 2
oof["Soft vote | full"] = soft_full

dev_hours = (dev["t_start"].max() - dev["t_start"].min()) / 3600
normal_scores = soft_full[kind_dev == "normal"]

rows = {}
for fpr_target in [0.02, 0.01, 0.005, 0.002, 0.001]:
    thr = np.quantile(normal_scores, 1 - fpr_target)
    r = score_report(y_dev, soft_full, kind_dev, threshold=thr)
    r["threshold"] = thr
    r["false_alerts_per_day"] = r["fpr"] * (kind_dev == "normal").sum() / dev_hours * 24
    rows[f"target fpr {fpr_target:.1%}"] = r

thresholds = pd.DataFrame(rows).T[["threshold", "recall_other", "fpr",
                                   "false_alerts_per_day", "recall_flood", "precision"]]
thresholds.round(4)

,threshold,recall_other,fpr,false_alerts_per_day,recall_flood,precision
target fpr 2.0%,0.1602,0.9565,0.0204,48.9584,1.0000,0.9893
target fpr 1.0%,0.2324,0.9420,0.0101,24.1613,1.0000,0.9947
target fpr 0.5%,0.6460,0.9420,0.0056,13.3523,1.0000,0.9971
target fpr 0.2%,0.6638,0.9130,0.0021,5.0866,1.0000,0.9989
target fpr 0.1%,0.6848,0.8841,0.0011,2.5433,0.9999,0.9994


In [21]:
thresholds.round(4).to_csv("../../docs/results/threshold_selection_dev.csv")

In [23]:
from apt_detection.device_telemetry.features import build_baseline, link_parents, parent_child_pairs

INT = Path("../../data/interim/device_telemetry")
proc1 = pd.read_parquet(INT / "phase1_process_table.parquet")
baseline = build_baseline(proc1)
baseline_pairs = set(parent_child_pairs(proc1, link_parents(proc1)))
print("baseline programs:", len(baseline["counts"]), " pairs:", len(baseline_pairs))

baseline programs: 251  pairs: 401


In [24]:
import json, joblib

FINAL_THRESHOLD = float(thresholds.loc["target fpr 0.2%", "threshold"])

final_config = {
    "feature_set": "full",
    "features": FEATS,
    "ensemble": "soft vote: mean of LightGBM and Extra Trees attack probabilities",
    "threshold": FINAL_THRESHOLD,
    "threshold_rule": "0.2% false-positive target on development out-of-fold scores",
    "novelty_model": "One-Class SVM (log1p + standard scaling), trained on normal processes only",
    "novelty_cutoff_quantile": 0.99,
    "cutoff_timestamp_utc": "2023-12-03 12:00",
    "random_state": 42,
}

X_full, y_full = dev[FEATS], dev["label"]
lgbm_final = make_lgbm().fit(X_full, y_full)
et_final = SUPERVISED["Extra Trees"]().fit(X_full, y_full)
svm_final = UNSUPERVISED["One-Class SVM (log+scaled)"]().fit(X_full[kind_dev == "normal"])
final_config["novelty_cutoff"] = float(np.quantile(
    -svm_final.score_samples(X_full[kind_dev == "normal"]), 0.99))

MODELS = Path("../../models/device_telemetry")
MODELS.mkdir(parents=True, exist_ok=True)
joblib.dump(lgbm_final, MODELS / "lightgbm.joblib")
joblib.dump(et_final, MODELS / "extra_trees.joblib")
joblib.dump(svm_final, MODELS / "one_class_svm.joblib")
joblib.dump(baseline, MODELS / "baseline_programs.joblib")
joblib.dump(baseline_pairs, MODELS / "baseline_pairs.joblib")

Path("../../configs/device_telemetry_final.json").write_text(json.dumps(final_config, indent=2))
print("threshold:", round(FINAL_THRESHOLD, 4))
print("saved:", sorted(p.name for p in MODELS.iterdir()))

threshold: 0.6638
saved: ['baseline_pairs.joblib', 'baseline_programs.joblib', 'extra_trees.joblib', 'lightgbm.joblib', 'one_class_svm.joblib']
